# Цаг агаарын заалтын архив

Ус цаг уур, орчны шинжилгээний газрын ажиглалтыг татаж **ХОЁР өөр
Feature Service** рүү бичнэ.

| Зорилт | Портал | Хамрах хүрээ | Станц |
|---|---|---|---|
| Улс | `arcgis.ubhub.mn` | улс даяар | ~317 |
| Нийслэл | `environment.ub.gov.mn` | Нийслэл аймаг | 7 |

Эх сурвалжаас **нэг л удаа** татаж, хоёуланд нь хуваарилна — API руу
хоёр дахин хандах шаардлагагүй.

**Яагаад архив хэрэгтэй вэ.** `weather.gov.mn`-ий API нь ЗӨВХӨН хамгийн
сүүлийн заалтыг буцаадаг: өдөр, сар, жилийн цуваа авах зам байхгүй.
Тиймээс хандлага харах ганц арга бол заалтыг тухай бүрд нь өөрсдөө
хадгалж эхлэх.

**Энэ notebook нь `arcgis.ubhub.mn` дээр ажиллана.** Тэр порталын
давхаргад `GIS("home")`-ээр нэвтэрнэ; нөгөө порталд тусад нь нэвтрэх
шаардлагатай (доорх «Нэвтрэлт» хэсгийг үз).

## Тохиргоо

Зорилт бүрд гурван зүйл: давхаргын хаяг, хамрах хүрээ, нэвтрэх арга.

In [ ]:
import json
import urllib.request

# `datetime` нэрийг модультай нь ЗӨРЧҮҮЛЭХГҮЙ. `from datetime import
# datetime` гэвэл орчин дотор `import datetime` хаа нэгтээ хийгдсэн үед
# нэр нь модуль руу буцаж холбогдож "module has no attribute 'now'"
# гэсэн алдаа өгдөг.
import datetime as dt

from arcgis.gis import GIS
from arcgis.features import FeatureLayer

API = "https://weather.gov.mn/api/get"

# Хостын урд WAF сууж, танил бус агентыг 403-аар хаадаг.
# Энэ толгойгүй бол хүсэлт чимээгүй унана.
UA = {"User-Agent": "Mozilla/5.0"}

CAPITAL = "Нийслэл"

# ─────────────────────────────────────────────────────────────────────
#  ХОЁР ЗОРИЛТ
#
#  home = True   → GIS("home"), энэ notebook-ийн портал (нууц үг хэрэггүй)
#  home = False  → тусдаа нэвтэрнэ, нууц үг нь доорх файлаас уншигдана
#
#  ⚠ environment.ub.gov.mn нь ӨӨРӨӨ ГАРЫН ҮСЭГ ЗУРСАН гэрчилгээтэй тул
#    verify_cert=False заавал — эс тэгвээс SSL алдаа өгнө.
# ─────────────────────────────────────────────────────────────────────
TARGETS = [
    {
        "name": "Улс даяар",
        "scope": "all",
        "home": True,
        "url": (
            "https://arcgis.ubhub.mn/arcgis/rest/services"
            "/Hosted/Tsag_agaar_arhiv_uls/FeatureServer/0"
        ),
    },
    {
        "name": "Нийслэл",
        "scope": "capital",
        "home": False,
        "portal": "https://environment.ub.gov.mn/gis",
        "verify_cert": False,
        "url": (
            "https://environment.ub.gov.mn/hosting/rest/services"
            "/Hosted/Tsag_agaar_arhiv/FeatureServer/0"
        ),
    },
]

## Нэвтрэлт

`GIS("home")` нь notebook ажиллаж буй порталд автоматаар нэвтэрдэг —
нууц үг хэрэггүй. Нөгөө порталд бүртгэл шаардлагатай.

⚠ **Нууц үгийг notebook дотор БҮҮ БИЧ.** Notebook-ийг харах эрхтэй хүн
бүр түүнийг уншина, мөн Snapshot бүрд хадгалагдана. Оронд нь дараах
файлыг НЭГ УДАА үүсгэнэ (энэ нүдийг ажиллуулаад дараа нь агуулгыг нь
устгана):

```python
import json, pathlib
pathlib.Path("/arcgis/home/weather-archive-secrets.json").write_text(
    json.dumps({"username": "…", "password": "…"}), encoding="utf-8"
)
```

Файл нь таны хувийн ажлын талбарт үлдэх бөгөөд notebook-ийн кодод
орохгүй. Файл байхгүй бол хоёр дахь зорилт АЛГАСАГДАНА — эхнийх нь
хэвийн ажиллана.

In [ ]:
SECRETS = "/arcgis/home/weather-archive-secrets.json"


def connect(target):
    """Зорилтын давхаргыг нээнэ. Боломжгүй бол `None`."""
    if target["home"]:
        return FeatureLayer(target["url"], GIS("home"))

    try:
        with open(SECRETS, encoding="utf-8") as f:
            cred = json.load(f)
    except FileNotFoundError:
        print(f"  [{target['name']}] нууц үгийн файл алга — алгаслаа")
        return None

    gis = GIS(
        target["portal"],
        cred["username"],
        cred["password"],
        verify_cert=target.get("verify_cert", True),
    )
    return FeatureLayer(target["url"], gis)

## Эх сурвалжаас татах

In [ ]:
def get(path):
    req = urllib.request.Request(f"{API}/{path}", headers=UA)
    with urllib.request.urlopen(req, timeout=90) as r:
        return json.loads(r.read().decode("utf-8"))


# Станцын бүртгэл нь координат, өндөршлийг өгнө; ажиглалтад тэдгээр байхгүй
registry = {
    s["sid"]: s
    for s in get("obs/aimags")["aimag_sum"]
    if s.get("lat") is not None and s.get("lon") is not None
}

observations = get("obs/data/aws")["stationAWS"]
print(f"{len(observations)} станцын заалт ирлээ, бүртгэлд {len(registry)} станц")

## Бичлэг бэлдэх ба архивлах

Зорилт бүрд дараалал нь ижил:

1. Тухайн давхаргаас станц бүрийн **хамгийн сүүлийн `obs_date`**-ыг асуух
2. Түүнээс хойшхи заалтыг л бичлэг болгох
3. Нэмэх

⚠ **1-р алхмыг алгасч БОЛОХГҮЙ.** API нь станц шинэчлэх хүртэл ИЖИЛ
заалтыг буцаасаар байдаг (зарим станц 6 цаг тутам л мэдээлдэг).
Шалгахгүй бол цаг тутам бүх станц дахин бичигдэж, архив хэдхэн сард
ашиглах боломжгүй болно.

In [ ]:
def num(v):
    """Тоо мөн эсэх — эх сурвалж хоосныг null эсвэл мөрөөр өгдөг."""
    return v if isinstance(v, (int, float)) and not isinstance(v, bool) else None


def epoch_ms(iso):
    """ISO мөрийг epoch миллисекунд болгоно. ArcGIS огноог ингэж хадгална."""
    return int(dt.datetime.fromisoformat(iso.replace("Z", "+00:00")).timestamp() * 1000)


def latest_per_station(layer):
    """Давхарга дээрх станц бүрийн хамгийн сүүлийн ажиглалтын мөч."""
    out = {}
    res = layer.query(
        where="1=1",
        group_by_fields_for_statistics="sid",
        out_statistics=[{
            "statisticType": "max",
            "onStatisticField": "obs_date",
            "outStatisticFieldName": "last_obs",
        }],
        return_geometry=False,
    )
    for f in res.features:
        a = f.attributes
        if a.get("sid") is not None and a.get("last_obs") is not None:
            out[a["sid"]] = a["last_obs"]
    return out


def build(scope, latest, now_ms):
    """Шинэ заалтуудыг ArcGIS бичлэг болгоно."""
    adds = []
    for r in observations:
        st = registry.get(r["sid"])
        if not st:
            continue
        if scope == "capital" and st.get("aimag_name") != CAPITAL:
            continue

        at = epoch_ms(r["obs_date"])
        seen = latest.get(r["sid"])
        if seen is not None and at <= seen:
            continue

        # pst дээр 0 нь ХЭМЖИГДЭЭГҮЙН тэмдэг — станцын түвшний даралт
        # хэзээ ч тэг болохгүй (сүлжээний бодит доод утга 764 гПа)
        pst = num(r.get("pst"))
        if pst == 0:
            pst = None

        adds.append({
            "geometry": {
                "x": st["lon"],
                "y": st["lat"],
                "spatialReference": {"wkid": 4326},
            },
            "attributes": {
                "sid": r["sid"],
                "name": st.get("sta_name") or st.get("sum_name") or "",
                "place": st.get("sum_name") or "",
                "aimag": st.get("aimag_name") or "",
                "elev": num(st.get("elev")),
                "lat": st["lat"],
                "lon": st["lon"],
                "obs_date": at,
                "logged_at": now_ms,
                "ttt": num(r.get("ttt")),
                "ttt_feels": num(r.get("ttt_feels")),
                "ff": num(r.get("ff")),
                "pst": pst,
                "nh": num(r.get("nh")),
                "wind_speed": num(r.get("wind_speed")),
                "wind_dir": num(r.get("wind_dir")),
                "precip": num(r.get("precip")),
                # Албан жагсаалтад байгаа ч хариултад үргэлж ирдэггүй
                "snow_depth": num(r.get("snow_depth")),
                "tmin": num(r.get("tmin")),
                "tmax": num(r.get("tmax")),
            },
        })
    return adds

## Ажиллуулах

Зорилт бүрийг ТУСАД нь боловсруулна: нэг нь уначихвал нөгөө нь
хэвийн үргэлжилнэ. Нэг портал унтарсан, нууц үг хуучирсан зэрэг нь
нөгөө архивыг зогсоох ёсгүй.

In [ ]:
now_ms = int(dt.datetime.now(dt.timezone.utc).timestamp() * 1000)
failed = []

for t in TARGETS:
    print(f"[{t['name']}]")
    try:
        layer = connect(t)
        if layer is None:
            continue

        latest = latest_per_station(layer)
        print(f"  архивт {len(latest)} станцын бичлэг байна")

        adds = build(t["scope"], latest, now_ms)
        if not adds:
            print("  шинэ заалт алга")
            continue

        res = layer.edit_features(adds=adds)
        ok = [x for x in res.get("addResults", []) if x.get("success")]
        bad = [x for x in res.get("addResults", []) if not x.get("success")]
        print(f"  {len(ok)} заалт нэмэгдлээ")
        if bad:
            raise RuntimeError(f"{len(bad)} бичлэг нэмэгдсэнгүй: {bad[0]}")

    except Exception as e:
        # Алдааг НЭРЛЭЖ үлдээнэ, гэхдээ дараагийн зорилтыг зогсоохгүй
        print(f"  АЛДАА: {type(e).__name__}: {e}")
        failed.append(t["name"])

if failed:
    raise RuntimeError("Дараах зорилт амжилтгүй: " + ", ".join(failed))
print("Дууслаа")

## Хуваарьт оруулах

Энэ хүртэлх алхмууд нь notebook зөв ажиллаж байгааг БАТЛАХ зорилготой.
Хуваарьт оруулахгүй бол архив байгаа мөр дээрээ зогсоно.

1. Notebook-оо **хадгал**
2. Notebook доторх **Tasks** (цагийн икон) → **Create task**
3. Title → **Next** → *Repeat type* **Hour**, *interval* **1**,
   *On minute* **15**, *Ending date* **Never** → **Create**

**Давтамж.** Улсын архив цаг тутам бол жилд ~2.6 сая бичлэг (~0.6 GB).
3 цаг тутам болговол 873 мянга. Синоптик станцууд 3, 6 цаг тутам
мэдээлдэг тул алдагдах зүйл бага.

**Батлах шалгалт:** эхний ажиллуулалт заалтуудыг нэмнэ, дараа нь
`Kernel → Restart Kernel and Run All Cells` хийхэд хоёр зорилт хоёулаа
`шинэ заалт алга` гэж хэлэх ёстой.